# House Price Prediction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.impute  import SimpleImputer
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold , train_test_split
from skopt import BayesSearchCV

pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
train_data = pd.read_csv(r"train.csv")
train_data.head(5)

In [ ]:
train_data.isna().sum()

In [ ]:
test_data = pd.read_csv("test.csv")
test_data.head(5)

In [ ]:
test_data.shape

In [ ]:
test_data.isna().sum()

### Remove Duplicate values(check Id)

In [ ]:
duplicates = train_data['Id'].duplicated()
train_data[duplicates]

In [ ]:
train_data.drop_duplicates(inplace=True)

### Remove values with no Id

In [ ]:
no_id_data = train_data[train_data['Id'].isna()]
no_id_data

In [ ]:
train_data = train_data[train_data['Id'].notna()]
train_data.head(5)

### Target Variables 

In [ ]:
x = train_data.drop(columns = ['SalePrice' , 'Id'])
y = train_data['SalePrice']

#### Transform Y using yeo-jhonson transformation

In [ ]:
y_transformer = PowerTransformer(method='yeo-johnson' , standardize=True)
y_transformed = y_transformer.fit_transform(y.to_frame())

### Important Variables

In [ ]:
ordinal_cols = ['LotShape','Utilities','LandSlope','ExterQual','ExterCond',
                'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional',
                'FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']



nominal_cols = ['MSZoning','Street','Alley','Neighborhood','LotConfig','BldgType','Condition1','Condition2','HouseStyle','LandContour','RoofStyle','RoofMatl','Exterior1st' ,'Exterior2nd' ,
                'MasVnrType' , 'Foundation' , 'Heating' , 'GarageType' , 'MiscFeature' , 'SaleType' , 'SaleCondition', 'Electrical' , 'CentralAir']



impute_constant_cols = ['MSSubClass','MSZoning','Street','LotShape','LandContour','Utilities','LotConfig','LandSlope',
                        'Neighborhood','Condition1','Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st',
                        'Exterior2nd','ExterQual','ExterCond','Foundation','Heating','HeatingQC','CentralAir','Electrical',
                        'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual','Fireplaces','GarageCars',
                        'PavedDrive','SaleType','SaleCondition', 'Functional']


# cols whose na values can be filled with median
numerical_cols = ['MSSubClass','LotArea','LotFrontage','OverallQual','OverallCond','YearBuilt','YearRemodAdd',
                  'MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF','2ndFlrSF','LowQualFinSF',
                  'GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd'
                  ,'Fireplaces','GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','3SsnPorch'
                  ,'ScreenPorch','PoolArea','MiscVal','MoSold','YrSold']


# cols whose na values can be filled with default text values (like 'None' or 'No Garage')
# cols whose na values can be filled with 0
# (GarageYrBlt is kept here because if a house has no garage, a 0 is a safe placeholder before scaling/binarizing)
fill_default_cols = ['Alley','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType',
                        'GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature' ,'GarageYrBlt']


default_map = {
    'Alley' : 'No alley',
    'MasVnrType' : 'None',
    'BsmtQual' : 'No Basement',
    'BsmtCond' : 'No Basement',
    'BsmtExposure' : 'No Basement',
    'BsmtFinType1' : 'No Basement',
    'BsmtFinType2' : 'No Basement',
    'FireplaceQu' : 'No Fireplace',
    'GarageType' : 'No Garage',
    'GarageFinish' : 'No Garage',
    'GarageQual' : 'No Garage',
    'GarageCond' : 'No Garage',
    'GarageYrBlt' : 0,
    'PoolQC' : 'No Pool',
    'Fence' : 'No Fence',
    'MiscFeature' : 'None'
}


ordinal_categories = {
'LotShape' : ['Missing' , 'IR3' , 'IR2' , 'IR1' , 'Reg'],
'Utilities' : ['Missing' , 'ELO' , 'NoSeWa' , 'NoSewr', 'AllPub'],
'LandSlope' : ['Missing' , 'Sev' , 'Mod' , 'Gtl'],
'ExterQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'ExterCond' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtQual' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtCond' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtExposure' : ['No Basement' , 'No' , 'Mn' , 'Av' , 'Gd'],
'BsmtFinType1' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'BsmtFinType2' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'HeatingQC' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'KitchenQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Functional' :['Missing', 'Sal' , 'Sev' , 'Maj2' , 'Maj1' , 'Mod' , 'Min2' , 'Min1' , 'Typ'],
'FireplaceQu' : ['No Fireplace' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageFinish' : ['No Garage' , 'Unf' , 'RFn' , 'Fin'],
'GarageQual' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageCond' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'PavedDrive' : ['Missing' , 'N' , 'P' , 'Y'],
'PoolQC' : ['No Pool' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Fence' : ['No Fence' , 'MnWw' , 'GdWo' , 'MnPrv' , 'GdPrv']
}


### Fill Default value columns 

In [ ]:
for col in fill_default_cols:
    default_val = default_map.get(col)
    train_data[col] = train_data[col].fillna(default_val)
    test_data[col] = test_data[col].fillna(default_val)
    print(f"Default Values of {col} filled with {default_val}")
    print(train_data[col].unique())
    print(train_data[col].unique())

### Main Pipeline

In [ ]:
category_cols = ordinal_cols + nominal_cols
x[category_cols] = x[category_cols].astype(str)

cat_feature_idx = [x.columns.get_loc(col) for col in category_cols]

model = CatBoostRegressor(
    random_state=42,
    task_type='GPU'
)

### Hyper Parameter Tuning

In [ ]:
kf = KFold(n_splits=5 , random_state=42 , shuffle=True)

In [ ]:
param_distributions = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'depth': [4, 5, 6, 8, 10],
    'l2_leaf_reg': [1, 3, 5, 10, 20],
    'bagging_temperature': [0.0, 1.0, 5.0, 10.0],
    'border_count': [32, 64, 128, 254],
    'iterations': [500, 1000, 2000 , 3000 , 5000] 
}

In [ ]:
bayes_Search = BayesSearchCV(
    estimator= model,
    search_spaces= param_distributions,
    n_iter=50,
    cv = kf,
    verbose=100,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    random_state=42
)

### Training and Finding Best Model

In [ ]:
bayes_Search.fit(x,y_transformed , cat_features = cat_feature_idx)

In [ ]:
best_params = bayes_Search.best_params_
print(f"Best Parametres : {best_params}")

In [33]:
x_train , x_val , y_train , y_val = train_test_split(x  , y_transformed , test_size=0.2 , random_state=42)

final_model = CatBoostRegressor(
    **best_params,    # Stop if no improvement for 50 consecutive trees
    task_type='GPU',
    random_state=42,
    verbose=100
)


final_model.fit(
    x , y_transformed,
    cat_features=cat_feature_idx,
    verbose=100
)

0:	learn: 0.9931470	total: 83ms	remaining: 6m 54s
100:	learn: 0.5629107	total: 8.1s	remaining: 6m 33s
200:	learn: 0.4042704	total: 17.5s	remaining: 6m 58s
300:	learn: 0.3384969	total: 26.7s	remaining: 6m 56s
400:	learn: 0.3074334	total: 36.2s	remaining: 6m 55s
500:	learn: 0.2899155	total: 46.8s	remaining: 7m
600:	learn: 0.2765534	total: 57.2s	remaining: 6m 58s
700:	learn: 0.2676483	total: 1m 8s	remaining: 7m 1s
800:	learn: 0.2602540	total: 1m 20s	remaining: 7m
900:	learn: 0.2539838	total: 1m 30s	remaining: 6m 51s
1000:	learn: 0.2489070	total: 1m 40s	remaining: 6m 42s
1100:	learn: 0.2450558	total: 1m 50s	remaining: 6m 32s
1200:	learn: 0.2411034	total: 2m 2s	remaining: 6m 26s
1300:	learn: 0.2372894	total: 2m 13s	remaining: 6m 20s
1400:	learn: 0.2335219	total: 2m 24s	remaining: 6m 12s
1500:	learn: 0.2304917	total: 2m 36s	remaining: 6m 4s
1600:	learn: 0.2275809	total: 2m 48s	remaining: 5m 57s
1700:	learn: 0.2250700	total: 3m 1s	remaining: 5m 51s
1800:	learn: 0.2225436	total: 3m 13s	remaini

CatBoostRegressor(bagging_temperature=5.0, border_count=128, depth=6, iterations=5000, l2_leaf_reg=1, learning_rate=0.01, loss_function='RMSE', random_state=42, task_type='GPU', verbose=100)

In [ ]:
test_ids = test_data['Id']
test_data.drop(columns=['Id'] , inplace = True)

In [34]:
test_data[category_cols] = test_data[category_cols].astype(str)
transformed_predictions = final_model.predict(test_data)

actual_predictions = y_transformer.inverse_transform(transformed_predictions.reshape(-1,1))
rounded_actual_predictions = [ round(x,4) for x in actual_predictions.flatten()]

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


In [35]:
results = pd.DataFrame({
    "Id" : test_ids,
    "SalePrice" : rounded_actual_predictions
})

results.to_csv("result.csv" , index=False)